# Figure S1E — Summary ROC Curves (300-Patient Test Set)

Per-AE ROC curves with AUC values and operating points (training-set-optimized thresholds).

**Data sources** (under `figures/figures_data/figure 1/data`):
- `llama_maverick_1k_results.csv` — model probability predictions (same file as Fig 1B)
- `final_gold_standard_1k.csv` — patient-level gold standard labels
- `test_mrns.csv` — locked 298-patient test split
- `additional_negative_mrns.txt` — 2 RAG-filtered patients padded as all-negative (300 total)

Outputs are written to `figure 1/results/supp/`.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

%matplotlib inline

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
#   figures/
#   ├── figures_data/figure 1/data/   ← inputs (shared OneDrive data dir)
#   └── v1/figure 1/
#       ├── scripts/                  ← this notebook
#       └── results/supp/             ← output PDF + CSV
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

# Matched to Performance_Matrix_Fig1B: same model file, same locked split, same 300-patient test set
MODEL_FILE = DATA / "llama_maverick_1k_results.csv"
GOLD_FILE  = DATA / "final_gold_standard_1k.csv"
TEST_MRN_FILE = DATA / "test_mrns.csv"
ADDITIONAL_NEG_MRN_FILE = DATA / "additional_negative_mrns.txt"

OUT_PATH = RESULTS / "supp" / "ROC_Curve_S1E.pdf"
CSV_OUT  = RESULTS / "supp" / "ROC_Curve_S1E_results.csv"

for p in [MODEL_FILE, GOLD_FILE, TEST_MRN_FILE, ADDITIONAL_NEG_MRN_FILE]:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Results: {RESULTS / 'supp'}")

In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
THRESHOLDS = {
    "pneumonitis":           0.710,
    "adrenal_insufficiency": 0.810,
    "liver_toxicity":        0.010,
    "colitis":               0.710,
    "hyperthyroidism":       0.810,
    "hypothyroidism":        0.510,
}

TOXICITIES = ["liver_toxicity", "hypothyroidism", "pneumonitis",
              "colitis", "adrenal_insufficiency", "hyperthyroidism"]
assert set(TOXICITIES) == set(THRESHOLDS), "TOXICITIES out of sync with THRESHOLDS"

DISPLAY_NAMES = {
    "pneumonitis":           "Pneumonitis",
    "adrenal_insufficiency": "Adrenal insufficiency",
    "liver_toxicity":        "Liver toxicity",
    "colitis":               "Colitis",
    "hyperthyroidism":       "Hyperthyroidism",
    "hypothyroidism":        "Hypothyroidism",
}

COLORS = {
    "pneumonitis":           "#1f77b4",
    "adrenal_insufficiency": "#ff7f0e",
    "liver_toxicity":        "#2ca02c",
    "colitis":               "#d62728",
    "hyperthyroidism":       "#9467bd",
    "hypothyroidism":        "#8c564b",
}

In [ ]:
def _norm_mrn(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(8)

def _read_csv(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        for enc in ("latin-1", "cp1252"):
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except UnicodeDecodeError:
                continue
        raise

In [ ]:
def load_data():
    df_model = _read_csv(MODEL_FILE)
    df_model = df_model.rename(columns={
        "liver toxicity": "liver_toxicity",
        "adrenal insufficiency": "adrenal_insufficiency",
    })
    df_model["mrn"] = _norm_mrn(df_model["mrn"])

    df_gold = _read_csv(GOLD_FILE)
    mrn_col = next((c for c in ["MRN", "mrn", "MRN_STR"] if c in df_gold.columns), None)
    if mrn_col is None:
        raise ValueError("MRN column not found in gold standard")
    df_gold["MRN"] = _norm_mrn(df_gold[mrn_col])
    df_gold = df_gold.set_index("MRN")

    shared = sorted(set(df_model["mrn"]) & set(df_gold.index))
    scores = df_model[df_model["mrn"].isin(shared)].groupby("mrn")[TOXICITIES].max().sort_index()

    truth = pd.DataFrame(index=shared, columns=TOXICITIES, dtype=int)
    for mrn in shared:
        for tox in TOXICITIES:
            val = df_gold.at[mrn, tox] if tox in df_gold.columns else 0
            truth.at[mrn, tox] = 0 if pd.isna(val) else int(round(float(val)))
    truth = truth.loc[shared]

    test_df = pd.read_csv(TEST_MRN_FILE)
    test_col = next(c for c in test_df.columns if c.lower() in {"mrn", "mrn_str"})
    test_mrns = [m for m in _norm_mrn(test_df[test_col]) if m in scores.index]
    scores, truth = scores.loc[test_mrns], truth.loc[test_mrns]

    # ---- 2 patients filtered out at RAG retrieval: no evidence -> no LLM call -> ----
    # ---- negative for every toxicity. 0.0 is real pipeline output, not missing.  ----
    neg_mrns = [ln.strip().zfill(8) for ln in
                ADDITIONAL_NEG_MRN_FILE.read_text().splitlines() if ln.strip()]
    assert len(neg_mrns) == 2, f"Expected 2 MRNs, got {len(neg_mrns)}"
    for mrn in neg_mrns:
        if mrn in scores.index:
            continue
        zero = pd.DataFrame([[0.0] * len(TOXICITIES)], columns=TOXICITIES, index=[mrn])
        scores = pd.concat([scores, zero])
        truth = pd.concat([truth, zero.astype(int)])

    return scores, truth

scores, truth = load_data()
print(f"Patients: {len(scores)}")

In [ ]:
# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(3.7, 2.0))
ax.plot([0, 1], [0, 1], linestyle="--", color="#b0b0b0", linewidth=0.8, zorder=1)

roc_rows = []

for tox in TOXICITIES:
    y_true = truth[tox].values.astype(int)
    y_score = scores[tox].values.astype(float)

    if y_true.sum() == 0 or (y_true == 1).all():
        print(f"  {tox}: skipped (no variation in gold labels)")
        continue

    fpr, tpr, roc_thresholds = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    color, name = COLORS[tox], DISPLAY_NAMES[tox]
    ax.plot(fpr, tpr, color=color, linewidth=1.5, zorder=2,
            label=f"{name} (AUC={roc_auc:.2f})")

    thr = THRESHOLDS[tox]
    y_pred = (y_score >= (thr - 1e-10)).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    op_tpr = tp / (tp + fn) if (tp + fn) else 0
    op_fpr = fp / (fp + tn) if (fp + tn) else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    f1 = 2 * prec * op_tpr / (prec + op_tpr) if (prec + op_tpr) else 0

    ax.scatter(op_fpr, op_tpr, color=color, s=40, zorder=3, edgecolors="white",
               linewidths=0.5, label=f"{name} thr={thr:.2f}")

    roc_rows.append({
        "toxicity": tox, "toxicity_display": name, "auc": roc_auc,
        "threshold": thr, "operating_tpr": op_tpr, "operating_fpr": op_fpr,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "sensitivity": op_tpr, "specificity": 1 - op_fpr, "precision": prec, "f1": f1,
        "n_positive": int(y_true.sum()), "n_total": len(y_true),
    })

    print(f"  {tox}: AUC={roc_auc:.2f}, thr={thr:.2f}, TPR={op_tpr:.2f}, FPR={op_fpr:.2f}")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

for spine in ax.spines.values():
    spine.set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)

ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), bbox_transform=fig.transFigure,
          borderaxespad=0.3, frameon=True, framealpha=1.0, edgecolor="#cccccc",
          handlelength=1.2, handletextpad=0.4, labelspacing=0.35)

fig.subplots_adjust(left=0.10, right=0.62, top=0.97, bottom=0.15)
fig.savefig(OUT_PATH, format="pdf", dpi=450)
print(f"\nSaved: {OUT_PATH.name}")

roc_df = pd.DataFrame(roc_rows)
CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
roc_df.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT.name}")
plt.show()